In [ ]:
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
reference_points = np.load('/home/ray/projects/helicopter/assets/point_clouds/green_syma.npy')

In [ ]:
reference_mean = reference_points.mean(axis=0)

In [ ]:
reference_mean

In [ ]:
def plot_3d(points, window, marker_size=15, c=None, _floor_idxs = None, _wall_idxs = None, show_indices=True, center=True):
    plt.close('all')

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection='3d')

    if center:
        centered = points - np.mean(points, axis=0)
    else:
        centered = points
    x = centered[:, 0]
    y = centered[:, 1]
    z = centered[:, 2]

    if c is None:
        c = y

    ax.scatter(x, y, z, c=c, cmap='plasma', s=marker_size, depthshade=True)

    if show_indices:
        for i in range(len(centered)):
            # You can add a tiny offset to x, y, or z if you want the text floating slightly off the dot
            ax.text(x[i], y[i], z[i], str(i), size=8, color='black', zorder=10)

    if _floor_idxs is not None:
        ax.scatter(x[_floor_idxs], y[_floor_idxs], z[_floor_idxs], color='g', s=40, depthshade=True)
    if _wall_idxs is not None:
        ax.scatter(x[_wall_idxs], y[_wall_idxs], z[_wall_idxs], color='r', s=40, depthshade=True)

    ax.set_xlim(-window, window)
    ax.set_ylim(-window, window)
    ax.set_zlim(-window, window)

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    plt.show()

In [ ]:
%matplotlib ipympl

plot_3d(reference_points, window=0.1)

In [ ]:
floor_idxs = [32, 29, 30]
wall_idxs = [7, 8, 6]

In [ ]:
%matplotlib ipympl

plot_3d(points=reference_points, window=0.1, _floor_idxs=floor_idxs, _wall_idxs=wall_idxs)

In [ ]:
from scipy.spatial.transform import Rotation

In [ ]:
def get_normal_vector(arr, _idxs):
    subset = arr[_idxs]
    svd = np.linalg.svd(subset.T - np.mean(subset.T, axis=1, keepdims=True))
    left = svd[0]
    return left[:, -1]

In [ ]:
def register_orientation(points, _floor_idxs, _wall_idxs):
    up_target = np.array([0, 0, 1.0])
    right_target = np.array([0, -1.0, 0.0])

    up_normal = get_normal_vector(points, _floor_idxs)

    if np.dot(up_normal, up_target) < 0:
        up_normal = -up_normal

    axis_up = np.cross(up_normal, up_target)
    axis_up_norm = np.linalg.norm(axis_up)

    if axis_up_norm > 1e-6:
        angle_up = np.arctan2(axis_up_norm, np.dot(up_normal, up_target))
        up_quat = Rotation.from_rotvec((axis_up / axis_up_norm) * angle_up)
    else:
        up_quat = Rotation.identity()

    up_registered = up_quat.apply(points)

    right_normal = get_normal_vector(up_registered, _wall_idxs)

    if np.dot(right_normal, right_target) < 0:
        right_normal = -right_normal

    right_normal_xy = np.array([right_normal[0], right_normal[1], 0.0])
    right_normal_xy /= np.linalg.norm(right_normal_xy)

    cross_z = right_normal_xy[0] * right_target[1] - right_normal_xy[1] * right_target[0]
    dot_xy = np.dot(right_normal_xy, right_target)

    yaw_angle = np.arctan2(cross_z, dot_xy)

    right_quat = Rotation.from_rotvec(np.array([0, 0, yaw_angle]))

    fully_registered = right_quat.apply(up_registered)

    return fully_registered

In [ ]:
registered = register_orientation(reference_points, floor_idxs, wall_idxs)

In [ ]:
registered = registered - np.mean(registered, axis=0)

In [ ]:
%matplotlib ipympl

plot_3d(registered, window=0.1, center=False)

In [ ]:
# Adding vertical offset for skids (1.2 cm)
offset = 0.012 - registered[floor_idxs][:, 2].mean()

In [ ]:
offset

In [ ]:
registered[:, 2] += offset

In [ ]:
registered.mean(axis=0)

In [ ]:
np.save('/home/ray/projects/helicopter/assets/point_clouds/green_syma.npy', registered)